# UCB vs EI: adding the Upper Confidence Bound acquisition arm (Buchwald-Hartwig)

Adds a `qUpperConfidenceBound`/`UpperConfidenceBound` (beta=2.0) acquisition
arm alongside `LogExpectedImprovement`, for **GP-BO**, **PCV**, and
**BatchSelect**: `bo_puro`/`bo_puro_ucb`, `multiagente`/`multiagente_ucb`,
`batch_llm`/`batch_llm_ucb`. Uses `qwen3:30b`.

**Provenance note (see `experiments/README.md` for full detail):** this
notebook's own local run (below) uses `SEEDS = range(10)`, `N_ITER = 30`,
and writes `checkpoint_full_grid.pkl` / `all_results_full_grid.csv` (n=10).
The file actually required by `scripts/generate_figures.py` --
`all_results_full_grid_20seeds.csv` (n=20) -- is **not** produced by the
code below as currently saved; the original notebook's markdown states an
earlier version of this same notebook was run locally with `qwen2.5:7b` and
20 seeds to produce it, before being edited down to this smaller `qwen3:30b`
grid. Both files are still shipped in `experiments/`, and
`scripts/generate_figures.py` reads only the (validated, paper-matching)
n=20 file -- see `VALIDATION.md`.

Renamed from `dia12_ucb_vs_ei.ipynb`.

In [ ]:
from bayesllm.buchwald_hartwig import BuchwaldHartwigBenchmark

bench = BuchwaldHartwigBenchmark(
    data_path="data/Dreher_and_Doyle_input_data.xlsx",
    model="qwen3:30b",
)
print(f"Feature space dimensionality: {bench.n_dims}")
print(f"Emulator in-sample R2 (sanity check only): {bench.emulator.score(bench.X_scaled, bench.y_raw):.3f}")

## Method-specific orchestration (EI methods reuse `bench` directly; UCB variants and the batch dispatch are specific to this notebook)

In [ ]:
def run_pcv_iteration(X_bo, Y_bo, historial, bounds=None, max_rechazos=3, iteracion=0, seed=None, verbose=False):
    candidato_tensor = bench.propose_candidate(X_bo, Y_bo, bounds, seed=seed)
    return bench.run_pcv_deliberation(candidato_tensor, X_bo, Y_bo, historial,
                                       bounds=bounds, max_rechazos=max_rechazos,
                                       iteracion=iteracion, verbose=verbose)


def run_pcv_iteration_ucb(X_bo, Y_bo, historial, bounds=None, beta=2.0, max_rechazos=3,
                           iteracion=0, seed=None, verbose=False):
    candidato_tensor = bench.propose_candidate_ucb(X_bo, Y_bo, bounds, beta=beta, seed=seed)
    resultado = bench.run_pcv_deliberation(candidato_tensor, X_bo, Y_bo, historial,
                                            bounds=bounds, max_rechazos=max_rechazos,
                                            iteracion=iteracion, verbose=verbose)
    resultado['fuente'] = 'multiagente_ucb' if resultado['fuente'] == 'multiagent' else 'bo_fallback_ucb'
    return resultado


def run_batch_iteration(X_bo, Y_bo, historial, bounds=None, q=4,
                         modo_seleccion='llm', max_rechazos=3, iteracion=0,
                         verbose=False):
    """EI-acquisition batch iteration -- qLogNEI proposes a batch of q
    candidates, one is selected via 'llm' (Selector agent), 'random', or
    'top_acquisition', then deliberation as in PCV."""
    candidatos_tensor = bench.propose_batch(X_bo, Y_bo, bounds, q=q, seed=iteracion)

    if modo_seleccion == 'llm':
        candidatos_dict_list = [bench.tensor_to_dict(candidatos_tensor[i]) for i in range(q)]
        idx, selection_reasoning = bench.call_selector(candidatos_dict_list, historial, seed=iteracion)
    elif modo_seleccion == 'random':
        rng = __import__('numpy').random.default_rng(iteracion)
        idx = int(rng.integers(0, q))
        selection_reasoning = 'Random selection (control condition)'
    elif modo_seleccion == 'top_acquisition':
        valores = bench.compute_acquisition_values(candidatos_tensor, X_bo, Y_bo, bounds)
        idx = __import__('torch').argmax(valores).item()
        selection_reasoning = f'Highest acquisition value (control condition, value={valores[idx].item():.3f})'
    else:
        raise ValueError(f"Unknown modo_seleccion: {modo_seleccion}")

    candidato_seleccionado = candidatos_tensor[idx].unsqueeze(0)

    resultado = bench.run_pcv_deliberation(
        candidato_seleccionado, X_bo, Y_bo, historial,
        bounds=bounds, max_rechazos=max_rechazos, iteracion=iteracion, verbose=verbose
    )
    resultado['fuente'] = f"batch_{modo_seleccion}" if resultado['fuente'] == 'multiagent' else 'bo_fallback'
    resultado['selection_reasoning'] = selection_reasoning
    resultado['batch_index_selected'] = idx

    return resultado


def run_batch_iteration_ucb(X_bo, Y_bo, historial, bounds=None, q=4, beta=2.0,
                             max_rechazos=3, iteracion=0, verbose=False):
    """UCB-acquisition batch iteration -- candidate pool comes from
    qUpperConfidenceBound instead of qLogNEI; the Selector prompt/logic is
    unchanged (isolates the acquisition-function axis)."""
    candidatos_tensor = bench.propose_batch_ucb(X_bo, Y_bo, bounds, q=q, beta=beta, seed=iteracion)
    candidatos_dict_list = [bench.tensor_to_dict(candidatos_tensor[i]) for i in range(q)]
    idx, selection_reasoning = bench.call_selector(candidatos_dict_list, historial, seed=iteracion)
    candidato_seleccionado = candidatos_tensor[idx].unsqueeze(0)

    resultado = bench.run_pcv_deliberation(
        candidato_seleccionado, X_bo, Y_bo, historial,
        bounds=bounds, max_rechazos=max_rechazos, iteracion=iteracion, verbose=verbose
    )
    resultado['fuente'] = 'batch_llm_ucb' if resultado['fuente'] == 'multiagent' else 'bo_fallback_ucb'
    resultado['selection_reasoning'] = selection_reasoning
    resultado['batch_index_selected'] = idx
    return resultado


def run_one_iteration(method, X_bo, Y_bo, historial, iteracion, seed_base):
    """Dispatch a single BO iteration to the correct method-specific
    function. A distinct-but-reproducible sub-seed is derived per iteration
    (seed_base * 1000 + iteracion)."""
    iter_seed = seed_base * 1000 + iteracion

    if method == "bo_puro":
        return bench.run_bo_only_iteration(X_bo, Y_bo, bounds=bench.bounds, seed=iter_seed)

    elif method == "bo_puro_ucb":
        candidato_tensor = bench.propose_candidate_ucb(X_bo, Y_bo, bench.bounds, beta=2.0, seed=iter_seed)
        y_final = bench.objective(candidato_tensor)
        return {'x': candidato_tensor, 'y': y_final, 'fuente': 'bo_puro_ucb',
                'rechazos': 0, 'reasoning': '', 'log_rechazos': []}

    elif method == "multiagente":
        return run_pcv_iteration(
            X_bo, Y_bo, historial, bounds=bench.bounds, max_rechazos=3,
            iteracion=iteracion, seed=iter_seed, verbose=False
        )

    elif method == "multiagente_ucb":
        return run_pcv_iteration_ucb(
            X_bo, Y_bo, historial, bounds=bench.bounds, beta=2.0, max_rechazos=3,
            iteracion=iteracion, seed=iter_seed, verbose=False
        )
    elif method == "batch_llm":
        return run_batch_iteration(
            X_bo, Y_bo, historial, bounds=bench.bounds, q=4,
            modo_seleccion="llm", max_rechazos=3, iteracion=iteracion, verbose=False
        )

    elif method == "batch_llm_ucb":
        return run_batch_iteration_ucb(
            X_bo, Y_bo, historial, bounds=bench.bounds, beta=2.0, max_rechazos=3, iteracion=iteracion, verbose=False
        )

    else:
        raise ValueError(f"Unknown method: {method}")

## Seeds loop: experimental configuration and main run

In [ ]:
SEEDS = list(range(10))  # [0, 1, ..., 9]
N_ITER = 30

METHODS = [
    "bo_puro", "bo_puro_ucb",
    "multiagente", "multiagente_ucb",
    "batch_llm", "batch_llm_ucb",
]

all_results = []

In [ ]:
import time
import pickle
import torch
import pandas as pd

FULL_CHECKPOINT_PATH = "checkpoint_full_grid.pkl"

def run_full_grid(methods, seeds, n_iter, verbose=False, checkpoint_path=None):
    results = []
    for seed in seeds:
        X_init, Y_init, history_init = bench.generate_initial_design(seed)
        for method in methods:
            X_bo, Y_bo, historial = bench.clone_run_state(X_init, Y_init, history_init)
            for iteracion in range(n_iter):
                resultado = run_one_iteration(method, X_bo, Y_bo, historial, iteracion, seed_base=seed)
                x_new = resultado['x']
                y_new = resultado['y']
                X_bo = torch.cat([X_bo, x_new])
                Y_bo = torch.cat([Y_bo, y_new])
                historial.append({'descriptors': bench.tensor_to_dict(x_new), 'yield': y_new.item()})
                results.append({
                    'method': method, 'seed': seed, 'iteracion': iteracion,
                    'yield': y_new.item(), 'fuente': resultado['fuente'], 'rechazos': resultado['rechazos'],
                })
                if verbose:
                    print(f"[seed={seed}][{method}][iter={iteracion}] "
                          f"yield={y_new.item():.2f}% fuente={resultado['fuente']} rechazos={resultado['rechazos']}")
                if checkpoint_path is not None:
                    with open(checkpoint_path, "wb") as f:
                        pickle.dump(results, f)
    return results


t0 = time.time()
all_results = run_full_grid(METHODS, SEEDS, N_ITER, verbose=True, checkpoint_path=FULL_CHECKPOINT_PATH)
elapsed = time.time() - t0
print(f"\nDone. Total time: {elapsed/60:.1f} min ({elapsed/3600:.2f} h)")

df_all_results = pd.DataFrame(all_results)
df_all_results.to_csv("all_results_full_grid.csv", index=False)
df_all_results.head()

## Post-hoc analysis: UCB vs EI, and PCV/BatchSelect vs GP-BO, within this local n=10 run

In [ ]:
from scipy.stats import wilcoxon

def best_so_far(group):
    return group.sort_values('iteracion')['yield'].cummax()

df = df_all_results.copy()
df['best_so_far'] = df.groupby(['method', 'seed'], group_keys=False).apply(best_so_far)
final_best = df[df['iteracion'] == N_ITER - 1].pivot(index='seed', columns='method', values='best_so_far')

def compare(m1, m2):
    stat, p = wilcoxon(final_best[m1], final_best[m2])
    print(f"{m1:16s} vs {m2:16s}: p={p:.4f}  "
          f"(mean {m1}={final_best[m1].mean():.2f}%, mean {m2}={final_best[m2].mean():.2f}%)")

# UCB vs EI, within each architecture -- the central comparison of this exercise
compare('bo_puro', 'bo_puro_ucb')
compare('multiagente', 'multiagente_ucb')
compare('batch_llm', 'batch_llm_ucb')

# Bonus: does the multi-agent layer add anything over plain BO, for each acquisition function
compare('bo_puro', 'multiagente')
compare('bo_puro_ucb', 'multiagente_ucb')
compare('bo_puro', 'batch_llm')
compare('bo_puro_ucb', 'batch_llm_ucb')